видеосу жок, (только 13.09.2025 тушундурмосу).

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
# установить совместимую связки
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124 -q

In [3]:
import os
import glob
import torch
import torchaudio
import pandas as pd
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset, random_split

In [16]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ravdess-emotional-speech-audio' dataset.
Path to dataset files: /kaggle/input/ravdess-emotional-speech-audio


In [17]:
emotion_map = {
    1: "neutral", 2: "calm",    3: "happy",   4: "sad",
    5: "angry",   6: "fearful", 7: "disgust",  8: "surprised"
}

In [18]:
transform = nn.Sequential(
    torchaudio.transforms.MelSpectrogram(sample_rate=48000, n_mels=128, n_fft=1024, hop_length=512),
    torchaudio.transforms.AmplitudeToDB()
)
max_len = 500

In [19]:
class RAVDESSDataset(Dataset):
    def __init__(self, root_path, transform, max_len):
        self.transform = transform
        self.max_len = max_len
        self.audios = []

        for actor_folder in os.listdir(root_path):
            actor_path = os.path.join(root_path, actor_folder)
            if not os.path.isdir(actor_path):
                continue
            for fname in os.listdir(actor_path):
                if fname.endswith('.wav'):
                    parts = fname.replace('.wav', '').split('-')
                    emotion_id = int(parts[2]) - 1  # 0-индексация → 0..7
                    self.audios.append((os.path.join(actor_path, fname), emotion_id))

    def __len__(self):
        return len(self.audios)

    def __getitem__(self, idx):
        audio_path, emotion_id = self.audios[idx]
        waveform, sample_rate = torchaudio.load(audio_path)

        # Стерео → моно
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        if sample_rate != 48000:
            waveform = torchaudio.transforms.Resample(sample_rate, 48000)(waveform)

        spec = self.transform(waveform).squeeze(0)

        if spec.shape[1] > self.max_len:
            spec = spec[:, :self.max_len]
        else:
            spec = F.pad(spec, (0, self.max_len - spec.shape[1]))

        return spec, emotion_id


In [20]:
dataset = RAVDESSDataset(path, transform, max_len)

train_size = int(len(dataset) * 0.8)
test_size = len(dataset) - train_size

train_data, test_data = random_split(dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))

In [21]:
train = DataLoader(train_data, batch_size=32, shuffle=True)
test  = DataLoader(test_data,  batch_size=32)

In [27]:
class CheckAudioEmotion(nn.Module):
    def __init__(self):
        super().__init__()
        self.first = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.second = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 8)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        return self.second(self.first(x))

In [28]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [29]:
model = CheckAudioEmotion().to(device)

In [30]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

In [31]:
for epoch in range(300):
    model.train()
    total_loss = 0.0

    for x_batch, y_batch in train:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        y_pred = model(x_batch)
        loss = loss_fn(y_pred, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f'Эпоха {epoch + 1}, Потери: {total_loss:.2f}')

Эпоха 1, Потери: 74.27
Эпоха 2, Потери: 67.28
Эпоха 3, Потери: 64.53
Эпоха 4, Потери: 63.30
Эпоха 5, Потери: 62.28
Эпоха 6, Потери: 60.72
Эпоха 7, Потери: 60.87
Эпоха 8, Потери: 60.42
Эпоха 9, Потери: 60.46
Эпоха 10, Потери: 59.46
Эпоха 11, Потери: 59.36
Эпоха 12, Потери: 58.80
Эпоха 13, Потери: 59.42
Эпоха 14, Потери: 58.76
Эпоха 15, Потери: 57.68
Эпоха 16, Потери: 57.82
Эпоха 17, Потери: 57.39
Эпоха 18, Потери: 57.45
Эпоха 19, Потери: 57.34
Эпоха 20, Потери: 56.58
Эпоха 21, Потери: 57.43
Эпоха 22, Потери: 57.96
Эпоха 23, Потери: 56.91
Эпоха 24, Потери: 57.61
Эпоха 25, Потери: 56.44
Эпоха 26, Потери: 56.82
Эпоха 27, Потери: 56.03
Эпоха 28, Потери: 56.47
Эпоха 29, Потери: 56.30
Эпоха 30, Потери: 55.85
Эпоха 31, Потери: 55.82
Эпоха 32, Потери: 55.91
Эпоха 33, Потери: 55.58
Эпоха 34, Потери: 55.64
Эпоха 35, Потери: 55.62
Эпоха 36, Потери: 55.42
Эпоха 37, Потери: 55.36
Эпоха 38, Потери: 55.91
Эпоха 39, Потери: 54.68
Эпоха 40, Потери: 54.61
Эпоха 41, Потери: 53.95
Эпоха 42, Потери: 54.17
Э

In [32]:
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for x_batch, y_batch in test:  # предполагается, что test — это DataLoader
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        y_pred = model(x_batch)
        pred = torch.argmax(y_pred, dim=1)

        correct += (pred == y_batch).sum().item()
        total += y_batch.size(0)

accuracy = correct * 100 / total
print(f'Точность предположения модели: {accuracy:.2f}%')

Точность предположения модели: 67.71%


In [33]:
from google.colab import files

torch.save(emotion_map, 'labels_RAVDESS_EmotionalSpeechAudio.pth')
torch.save(model.state_dict(), 'model_RAVDESS_EmotionalSpeechAudio.pth')

files.download('labels_RAVDESS_EmotionalSpeechAudio.pth')
files.download('model_RAVDESS_EmotionalSpeechAudio.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>